# GSEM Bootstrapped Quadratic Scores

# Initialization

In [1]:
#%config SuppressWarnings.filterwarnings = "ignore"
## Set up
# Suppress Future Warnings
import warnings
#warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


## Import Packages
import os
import sys
from datetime import datetime

import pandas as pd
import numpy as np
import re # Regular Expressions

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as patches

# import seaborn as sns
# import plotly.express as px

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score, fowlkes_mallows_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

import scipy
from scipy.spatial.distance import cdist
from scipy.stats import norm, skew, kurtosis, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from joblib import Parallel, delayed

# Time functions
import time
 
# Suppress print() in functions
import contextlib
import io

def suppress_prints(func, *args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)

# Suppress prints
def suppress_print(func, *args, **kwargs):
    old_stdout = sys.stdout
    sys.stdout = open(os.devnull, "w")
    try:
        result = func(*args, **kwargs)
    finally:
        sys.stdout.close()
        sys.stdout = old_stdout
    return result

# warnings.filterwarnings("ignore", category=FutureWarning)
# warnings.filterwarnings(
#     "ignore",
#     message=".*Series.replace.*CategoricalDtype.*",
#     category=FutureWarning,
# )

import logging

logging.basicConfig(
    filename='debugGSEM_BQS.log',          # name of your log file
    filemode='w',                  # 'w' = overwrite each run, 'a' = append
    level=logging.INFO,            # INFO = normal logs, DEBUG = more detailed
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Select Random Seed
RandomSeed = 42
print(f'Random seed is set to {RandomSeed}')

# Folder with datasets
test_folder_path = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/Bootstrap Data/GaussK3_b10"
folder_path = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/Bootstrap Data/GSEM Gauss/gsemGaussK3_bootstrap10"

# Data Storage Location:
data_storage = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/Quadratic Scores"

Random seed is set to 42


## Import & Format Data

In [12]:
# Import Dataset
#dataset = pd.read_stata(f'/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/PsAID_GSEM_Clustering.dta') #Stata data geformat
OG_dataset = pd.read_csv(f'/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/PsAID_Mixture Models_Clusters.csv') # Stata data geformat T IN 'MixtureModels.ipynb'
columns_gp = [i for i in OG_dataset.columns if re.match(r'^gp\d{2}$',i)]
cluster_columns = [i for i in OG_dataset.columns if re.match(r'^clusterID_',i)]

#example_file = f'{test_folder_path}/gsemGaussK3_bootstrap100_b1.dta'
#example_file = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/Bootstrap Data/GSEM Poisson/gsemPoissonK7_bootstrap1000/gsemPoissonK7_bootstrap1000_b301_FAILED.dta" # Dealing with _FAILED datasets
example_file = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/Bootstrap Data/GSEM Weibull/gsemWeibullK5_bootstrap100/gsemWeibullK5_bootstrap100_b5.dta" # Weibull datasets
example_sample_dataset = pd.read_stata(example_file)
example_sample_dataset.head()

,respondentid,mm,gp01,gp02,gp03,gp04,gp05,gp06,gp07,gp08,...,gp12,psaid12,b5_pr1,b5_pr2,b5_pr3,b5_pr4,b5_pr5,b5_clusters,algorithm,bootstrap
0,10173314.0,T9,2,2,3,3,2,2,3,2,...,1,1.15,1.343462e-02,5.877506e-20,0.986565,2.133412e-07,2.077760e-12,3.0,gsemWeibullK5,5.0
1,10173314.0,T18,2,2,1,1,1,2,2,2,...,2,0.65,9.954586e-04,5.633591e-03,0.993371,2.144641e-08,2.009531e-13,3.0,gsemWeibullK5,5.0
2,10418205.0,Intake ronde,Extreem<center>10</center>,6,4,7,7,7,9,6,...,3,5.55,2.936939e-16,0.000000e+00,0.000000,5.655159e-01,4.344840e-01,4.0,gsemWeibullK5,5.0
3,10418205.0,Intake ronde,Extreem<center>10</center>,6,4,7,7,7,9,6,...,3,5.55,2.936939e-16,0.000000e+00,0.000000,5.655159e-01,4.344840e-01,4.0,gsemWeibullK5,5.0
4,10418205.0,Intake ronde,Extreem<center>10</center>,6,4,7,7,7,9,6,...,3,5.55,2.936939e-16,0.000000e+00,0.000000,5.655159e-01,4.344840e-01,4.0,gsemWeibullK5,5.0


In [13]:
def format_data(data, columns_gp):
    
    # Format to integers
    data['respondentid'] = data['respondentid'].astype('int')
    if 'bootstrap' in data.columns:
        data['bootstrap'] = data['bootstrap'].astype('int')

    cluster_columns = [i for i in data.columns if re.match(r'^clusterID_', i) or i.endswith('_clusters')]  #select columsn in format ClusterID_* (OG Dataset) or *_clusters (Bootstrap datasets)
    data[cluster_columns] = data[cluster_columns].astype('int')

    # Make sure cluster IDs start from 0
    for clusterID in cluster_columns:
        if data[clusterID].min() >=1:
            data[clusterID] = data[clusterID]-1
    
    ## Format MM
    # Map categories
    mm_categoriesA = ['Intake ronde', 'T3', 'T6', 'T9', 'T12', 'T18', 'T24', 'T36', 'T48', 'T60', 'T72', 'T84', 'T96', 'T108' ] # List of original categories
    mm_categoriesB = ['0', '3', '6', '9', '12', '18', '24', '36', '48', '60', '72', '84', '96', '108'] # List of new categories
    data['mm'] = data['mm'].replace(dict(zip(mm_categoriesA ,mm_categoriesB))) #replace original with new categories

    # Convert objects to integers
    data['mm'] = data['mm'].astype('int')
    assert data['mm'].dtype == 'int' #Check if the mm are indeed integers

    ### Correct gp data
    # Convert from category to object so we can make changes
    data[columns_gp] = data[columns_gp].astype('object')
    for col in columns_gp:
        assert (data[col].dtype) == 'object' # Assure if datatype is changed

    # Map answer categories
    # Mapping is based on the PSAID12 questionnaire answers, see PDF [PsAID12_vragenlijst.pdf].
    # 0 = good, 10 = bad
    # Convert from category to object so we can make changes
    data[columns_gp] = data[columns_gp].astype('object')
    for col in columns_gp:
        assert (data[col].dtype) == 'object' # Assure if datatype is changed
    gp_categoriesA = ['Extreem<center>10</center> ', 'Geen<center>probleem</center><center>0</center> ', 'Volledig<center>uitgeput</center><center>10</center> ',  'Geen<center>vermoeidheid</center><center>0</center> ', 'Extreem<center>probleem</center><center>10</center> ', 'Geen<center>0</center> ', 'Geen<center>moeilijkheden</center><center>0</center> ', 'Heel<center>goed</center><center>0</center> ', 'Extreem<center>10</center>', 'Geen<center>probleem</center><center>0</center> ', 'Extreme<center>moeilijkheden</center><center>10</center> ', 'Heel<center>slecht</center><center>10</center> '] 
    gp_categoriesB = ['10', '0','10','0', '10', '0', '0', '0', '10', '0', '10', '10'] #New category names
    data = data.replace(dict(zip(gp_categoriesA, gp_categoriesB))) #Replace old with new category names
    data[columns_gp] = data[columns_gp].apply(pd.to_numeric).astype(int) # Convert selection to to numeric datatypes

    # For Weibull sample datasets gp_min = 1 and gp_max = 10, convert to 0-10
    if 'algorithm' in data.columns:
       if (data['algorithm'][0].startswith('gsemWeibull')) and ((data[columns_gp].min().min() >=1) and (data[columns_gp].max().max()>=11)):
           data[columns_gp] = data[columns_gp]-1
    
    for col in columns_gp:
        assert (data[col].dtype) == 'int' # Assure if datatype is changed
        assert (data[col] >= 0 ).all(), "Values smaller than 0" #Check that the values range between 0-10
        assert (data[col] <= 10).all(), "Values exceed 10" #Check that the values range between 0-10 

    # Drop _flag parameters
    flag_col = [col for col in data.columns if col.endswith('flag')]
    data = data.drop(columns = flag_col)

    return data

In [14]:
OG_dataset = format_data(OG_dataset, columns_gp)
example_sample_dataset2 = format_data(example_sample_dataset, columns_gp)
example_sample_dataset2.head()

,respondentid,mm,gp01,gp02,gp03,gp04,gp05,gp06,gp07,gp08,...,gp12,psaid12,b5_pr1,b5_pr2,b5_pr3,b5_pr4,b5_pr5,b5_clusters,algorithm,bootstrap
0,10173314,9,1,1,2,2,1,1,2,1,...,0,1.15,1.343462e-02,5.877506e-20,0.986565,2.133412e-07,2.077760e-12,2,gsemWeibullK5,5
1,10173314,18,1,1,0,0,0,1,1,1,...,1,0.65,9.954586e-04,5.633591e-03,0.993371,2.144641e-08,2.009531e-13,2,gsemWeibullK5,5
2,10418205,0,9,5,3,6,6,6,8,5,...,2,5.55,2.936939e-16,0.000000e+00,0.000000,5.655159e-01,4.344840e-01,3,gsemWeibullK5,5
3,10418205,0,9,5,3,6,6,6,8,5,...,2,5.55,2.936939e-16,0.000000e+00,0.000000,5.655159e-01,4.344840e-01,3,gsemWeibullK5,5
4,10418205,0,9,5,3,6,6,6,8,5,...,2,5.55,2.936939e-16,0.000000e+00,0.000000,5.655159e-01,4.344840e-01,3,gsemWeibullK5,5


In [71]:
OG_dataset['clusterID_gsemGauss_k6'].value_counts()

clusterID_gsemGauss_k6
3    2143
2    1110
0     580
5     517
4     392
1     357
Name: count, dtype: int64

# Compute Quadratic Score

In [15]:
def compute_QS(data, byvar, cluster_col):
    # Reserver space for qs_matrix
    n,p = data.shape  # Num_rows, Num_columns
    labels = data[cluster_col] # Cluster assignments
    k = len(labels.unique()) # Number of clusters

    print(f'Computing QS for {cluster_col}')

    # Extract parameters
    cluster_size = labels.value_counts().sort_index().values # Array of cluster sizes
    cluster_sizefraction = labels.value_counts().sort_index().values/n # Fraction of the sample belonging to each cluster
    cluster_centers = data[byvar + [cluster_col]].groupby(cluster_col).mean()  # Centroid vector

    # ---- validity check ----
    # Check for singular mixture components. If <2 the cov matrix is undefined/dof<=0 -->  If singular the $inv(Sigma_k)$ can not be computed because det(Sigma_k) = 0.
    # Thus can't calculate Tn
    if np.any(cluster_size <2):
        Tn = np.nan
        tau = np.nan
        comments = f'cluster {np.where(cluster_size < 2)[0].tolist()} invalid'
        return Tn, tau, comments

    # Compute Quadratic Score qs
    X = data[byvar].to_numpy() #Convert psaid data to numpy array
    qs_matrix = np.full((n, k), np.nan)  # Reserve space for qs_matrix
    
    for k in sorted(labels.unique()):
    
        pi_k = cluster_sizefraction[k] # Fraction of cluster k
        mu_k = cluster_centers.loc[k].to_numpy() # Centers of cluster k

        # Compute scatter/covariance
        cluster_data = data.loc[labels == k, byvar]
        Sig_k = np.cov(cluster_data, rowvar=False, ddof=1) # Compute sample scatter = covariance matrix of cluster k

        # Check for linear dependencies --> Compute Rank of sample 
        # If the cov matrix is singular --> Ridge regularization to make invertible
        if np.linalg.matrix_rank(Sig_k) < len(byvar): 
            Sig_k += 1e-6 * np.eye(len(byvar))

        # Compute the quadratic term = 1/2 * (xᵢ - μₖ)ᵀ Σₖ⁻¹ (xᵢ - μₖ)
        diff = X - mu_k # Difference Vectors
        inv_Sig = np.linalg.inv(Sig_k) #Log DEeterminant
        quad = np.einsum('ij,jk,ik->i', diff, inv_Sig, diff) # Quadratic term = Mahalanobis distance squared # Einsum lets you define custom summation patterns

        # Compute Log Determinant of Sig_k
        # By using slogdet and ignoring sign, Sig_k is assumed positive-definite, which is usually fine if you regularize
        sign, logdet = np.linalg.slogdet(Sig_k)
 
        # Compute the Quadratice Scores for all observations in cluster k
        qs = np.log(pi_k)-0.5*logdet-0.5*quad
        qs_matrix[:,k] = qs # Store all qs in one matrix

    # Compute the Quadratic Smooth Score (QS) = Tₙ(θ) = (1/n) ∑ᵢ ∑ₖ τₖ(xᵢ; θ) · qs(xᵢ, θₖ)
    # Compute the softmax weights with Softmax transformation --> τₖ(xᵢ; θ) = τₖ(xᵢ; θ) = exp(qs(xᵢ, θₖ)) / ∑ⱼ exp(qs(xᵢ, θⱼ))
    # Softmax ensures the scores are "probability like"
    max_qs = np.max(qs_matrix, axis=1, keepdims=True)  # For numerical stability, subtract the max per row
    exp_qs = np.exp(qs_matrix - max_qs)
    tau = exp_qs / exp_qs.sum(axis=1, keepdims=True)  # Softmax weights

    # Compute  QS
    Tn = np.mean(np.sum(tau * qs_matrix, axis=1))

    comments = ""

    return Tn, tau, comments, cluster_size

In [16]:
QS = []
# Run fol ALL DATA
for cluster in cluster_columns:
    Tn, _, comments, cluster_size = compute_QS(OG_dataset[columns_gp+[cluster]], columns_gp, cluster)
    QS.append({'K': cluster,'QS': Tn, 'QS_comment': comments, 'Cluster Size': cluster_size})

# # TEST WITH SAMPLE DATA
# Tn, _, comments, cluster_size = compute_QS(example_sample_dataset[columns_gp+['b1_clusters']], columns_gp, 'b1_clusters')
# QS.append({'K': 'b1_clusters','QS': Tn, 'QS_comment': comments, 'Cluster Size': cluster_size})

#Store Data
#Mark dataset with computation date.time
timestamp = datetime.now().strftime("%Y%m%d_%H:%M:%S")
qs_results = pd.DataFrame(QS)
qs_results['timestamp'] = timestamp

# Store
file_name = f'PsAID_QS_MixtureModels_{timestamp}'

# Ensure the folder exists
os.makedirs(data_storage, exist_ok=True)

# Save
qs_results.to_csv(f'{data_storage}/{file_name}.csv', index=False)
print(f' Dataset saved to {data_storage}/{file_name}.csv')

qs_results

Computing QS for clusterID_gsemGauss_k3
Computing QS for clusterID_gsemGauss_k4
Computing QS for clusterID_gsemGauss_k5
Computing QS for clusterID_gsemGauss_k6
Computing QS for clusterID_gsemGauss_k7
Computing QS for clusterID_gsemPoisson_k3
Computing QS for clusterID_gsemPoisson_k4
Computing QS for clusterID_gsemPoisson_k5
Computing QS for clusterID_gsemPoisson_k6
Computing QS for clusterID_gsemPoisson_k7
Computing QS for clusterID_gsemWeibull_k3
Computing QS for clusterID_gsemWeibull_k4
Computing QS for clusterID_gsemWeibull_k5
Computing QS for clusterID_gsemWeibull_k6
Computing QS for clusterID_gsemWeibull_k7
 Dataset saved to /Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/Quadratic Scores/PsAID_QS_MixtureModels_20260731_18:01:45.csv


,K,QS,QS_comment,Cluster Size,timestamp
0,clusterID_gsemGauss_k3,-9.369083,,"[1467, 952, 2680]",20260731_18:01:45
1,clusterID_gsemGauss_k4,-9.111620,,"[2377, 949, 498, 1275]",20260731_18:01:45
2,clusterID_gsemGauss_k5,-8.890018,,"[313, 659, 2240, 1161, 726]",20260731_18:01:45
3,clusterID_gsemGauss_k6,-8.770362,,"[580, 357, 1110, 2143, 392, 517]",20260731_18:01:45
4,clusterID_gsemGauss_k7,-8.700175,,"[454, 353, 2092, 1031, 339, 561, 269]",20260731_18:01:45
5,clusterID_gsemPoisson_k3,-8.397822,,"[1375, 1907, 1817]",20260731_18:01:45
6,clusterID_gsemPoisson_k4,-7.610108,,"[1447, 979, 1346, 1327]",20260731_18:01:45
7,clusterID_gsemPoisson_k5,-7.394812,,"[679, 1003, 746, 1271, 1400]",20260731_18:01:45
8,clusterID_gsemPoisson_k6,-6.851275,,"[1169, 591, 690, 699, 882, 1068]",20260731_18:01:45
9,clusterID_gsemPoisson_k7,-6.420445,,"[811, 632, 926, 1058, 543, 630, 499]",20260731_18:01:45


In [17]:
def compute_ARI(OG_data, sample_data, byvar, cluster_col):

    # Select Matching OG_data clustering
    matched_clusterID = re.match(r"(.+)K(\d+)$", sample_data['algorithm'][0])
    print(matched_clusterID)
    if matched_clusterID:
        prefix, k = matched_clusterID.groups()
        OG_clusterID = f"clusterID_{prefix}_k{k}"
        print(OG_clusterID)
    else:
        raise ValueError(f"Algorithm name '{sample_data['algorithm'][0]}' does not match expected pattern")
    
    OG_labels = OG_data[OG_clusterID] # Reference cluster assignments
    X_OG = OG_data[byvar].to_numpy()
    OG_n = OG_data.shape[0]
    sample_n = sample_data.shape[0]

    # Extract bootstrap cluster descriptions
    sample_labels = sample_data[cluster_col] # Cluster assignments
    k = len(sample_labels.unique()) # Number of clusters
    cluster_centers = sample_data[byvar + [cluster_col]].groupby(cluster_col).mean()  # Centroid vector
    cluster_size = sample_labels.value_counts().sort_index().values # Array of cluster sizes
    cluster_sizefraction = sample_labels.value_counts().sort_index().values/sample_n # Fraction of the sample belonging to each cluster

    # ---- validity check ----
    # Check for singular mixture components. If <2 the cov matrix is undefined/dof<=0 -->  If singular the $inv(Sigma_k)$ can not be computed because det(Sigma_k) = 0.
    # Thus can't calculate tau
    if np.any(cluster_size <2):
        sample_ARI = np.nan
        return sample_ARI

    # Reserve storage space
    qs_matrix = np.full((OG_n, k), np.nan)
    
    # Probabilites of original data belonging to bootstrap cluster k
    for k in sorted(sample_labels.unique()):

        cluster_data = sample_data.loc[sample_labels == k, byvar]
        
        # Compute scatter/covariance
        Sig_k = np.cov(cluster_data, rowvar=False, ddof=1) # Compute sample scatter = covariance matrix of cluster k
        if np.linalg.matrix_rank(Sig_k) < len(byvar): # If not full rank
            Sig_k += 1e-6 * np.eye(len(byvar))        # Ridge Regularization

        mu_k = cluster_centers.loc[k].to_numpy() # Centers of cluster k
        pi_k = cluster_sizefraction[k] # Cluster size fraction

        # Compute the quadratic term = 1/2 * (xᵢ - μₖ)ᵀ Σₖ⁻¹ (xᵢ - μₖ)
        diff = X_OG - mu_k # Difference Vectors
        inv_Sig = np.linalg.inv(Sig_k) #Log DEeterminant
        quad = np.einsum('ij,jk,ik->i', diff, inv_Sig, diff) # Quadratic term = Mahalanobis distance squared # Einsum lets you define custom summation patterns

        # Compute Log Determinant of Sig_k
        # By using slogdet and ignoring sign, Sig_k is assumed positive-definite, which is usually fine if you regularize
        sign, logdet = np.linalg.slogdet(Sig_k)
 
        # Compute the Quadratice Scores for all observations in cluster k
        qs = np.log(pi_k)-0.5*logdet-0.5*quad
        qs_matrix[:,k] = qs # Store all qs in one matrix

    # Softmax transformation to ensure scores are 'probability like'
    max_qs = np.max(qs_matrix, axis=1, keepdims=True)  # For numerical stability, subtract the max per row
    exp_qs = np.exp(qs_matrix - max_qs)
    tau = exp_qs / exp_qs.sum(axis=1, keepdims=True)  # Softmax weights = probabilities

    # Assign OG_data to the most likely bootstrap cluster
    projected_labels = tau.argmax(axis=1)

    # Compute Ari against original labels
    sample_ARI = adjusted_rand_score(OG_labels, projected_labels)

    # ---- per-cluster stability (Jaccard best-match, Hennig 2007) ----
    # For each ORIGINAL cluster, find the projected bootstrap cluster it best
    # overlaps with and record the Jaccard similarity. Reported per original
    # cluster ID so it can be aggregated across bootstraps downstream.
    per_cluster_jaccard = {}
    OG_labels_arr = OG_labels.to_numpy()
    for og_c in np.unique(OG_labels_arr):
        A = OG_labels_arr == og_c
        best = 0.0
        for proj_c in np.unique(projected_labels):
            B = projected_labels == proj_c
            union = np.sum(A | B)
            best = max(best, np.sum(A & B) / union if union else 0.0)
        per_cluster_jaccard[og_c] = best

    return OG_clusterID, sample_ARI, per_cluster_jaccard

In [51]:
def bootstrap_iteration(OG_data, cluster_columns, file):
    # Turn on Filter Warnings
    warnings.filterwarnings(
        "ignore",
        message=".*Series.replace.*CategoricalDtype.*",
        category=FutureWarning,
    )

    # Enable Logging
    import logging
    logging.basicConfig(
        filename='debug.log',
        level=logging.INFO,
        format='%(asctime)s - [%(processName)s] - %(levelname)s - %(message)s',
        force=True  # ensures every process configures logging
    )
    logging.info(f"file name received {file}")

    # Cant compute 
    if file.endswith("_FAILED.dta"):
        # Construct OG_cluster ID
        basename = os.path.splitext(os.path.basename(file))[0] # Extract just the base name without extension
        matched_OG_clusterID = re.match(r"(.+)K(\d+)_.*", basename) # Match pattern
        matched_clusterID = re.match(r".*(b\d+_).*", basename)
        if matched_OG_clusterID and matched_clusterID :
            OG_prefix, k = matched_OG_clusterID.groups()
            OG_clusterID = f"clusterID_{OG_prefix}_k{k}"
            prefix = matched_clusterID.group(1)
            clusterID = f'{prefix}clusters'
        else: 
            raise ValueError(f"Filename '{filename}' does not match expected pattern")

        # Other results are non-existing
        sample_QS = np.nan
        sample_tau = np.nan
        sample_comment = f'no clustering available'
        sample_ARI = np.nan
        sample_per_cluster_jaccard = {}
        print("error")

    else:
        # Import Dataset
        sample_data = pd.read_stata(f'{file}') # Import dataset
       # print(sample_data.head())
        columns_gp = [i for i in sample_data.columns if re.match(r'^gp\d{2}$',i)]
        clusterID = [col for col in sample_data.columns if col.endswith('_clusters')][0]
        
        # Format Dataset
        sample_data = format_data(sample_data, columns_gp)
        b = sample_data['bootstrap']
    
        # Compute sample QS
        sample_QS, sample_tau, sample_comment, cluster_size = suppress_print(compute_QS, sample_data, columns_gp, clusterID)
        #sample_QS, sample_tau, sample_comment, cluster_size = compute_QS(sample_data, columns_gp, clusterID)
        
        # Compute ARI
        OG_clusterID, sample_ARI, sample_per_cluster_jaccard = suppress_print(compute_ARI, OG_data[columns_gp+cluster_columns], sample_data[columns_gp+[clusterID, 'algorithm']], columns_gp, clusterID)
        # OG_clusterID, sample_ARI = compute_ARI(OG_data[columns_gp+cluster_columns], sample_data[columns_gp+[clusterID, 'algorithm']], columns_gp, clusterID)

    return OG_clusterID, sample_QS, sample_tau, sample_comment, clusterID, sample_ARI, sample_per_cluster_jaccard

In [52]:
def compute_BQS_ARI_inParallel(folder_path, OG_dataset, cluster_columns):
    logging.info(f"folderpath name received {folder_path}")

    start = time.time()
    
    # Count files only (ignore subfolders and files that are not .dta)
    n_bootstrap = sum(
        1 for f in os.listdir(folder_path) 
        if (os.path.isfile(os.path.join(folder_path, f)) and f.endswith(".dta"))
    )
    print(f'n_bootstrap: {n_bootstrap}')
    
    results = Parallel(n_jobs=-1, backend="loky", verbose=5)(  #threading if printing
        delayed(bootstrap_iteration)(OG_dataset, cluster_columns, os.path.join(folder_path, filename))
        for filename in os.listdir(folder_path)
        if os.path.isfile(os.path.join(folder_path, filename)) and not filename.startswith(".")
    )
    
    # Unpack results
    OG_clusterID_list, qs_list, tau_list, comment_list, clusterIDs, ARI_list, jaccard_list = map(list, zip(*results))
    #qs_list, tau_list, comment_list, clusterIDs, ARI_list = zip(*results)
    #print(set(qs_list))
    #print(jaccard_list)
    
    # Verify & Select OG_clusterID name
    assert all(x == OG_clusterID_list[0] for x in OG_clusterID_list), f"Not all OG_clusterIDs are the same!"
    OG_clusterID = OG_clusterID_list[0] # If the assert passes, pick the single value
    
    # ---- validity check ----
    # If +> 25% of the runs are invalid, discard the discard method.
    qs_array = np.array(qs_list) 
    n_nan = np.sum(np.isnan(qs_array))  # Count missing sample_qs values
    proportion_empty = n_nan / n_bootstrap * 100   # Convert # missing values to %
    if proportion_empty > 24: 
        BQS = np.nan
        results_dict = {
            'K': OG_clusterID,
            'ARI': np.nan,
            'QS_mean': np.nan,
            'L_ci': np.nan,
            'U_ci': np.nan,
            'all_QS': qs_array,
            '%-discarded': proportion_empty,
            'duration': np.nan,
            'b': n_bootstrap,
            'Jaccard': np.nan
        }
        return clusterID, BQS, results_dict, jaccard_list
        
    # Compute BQS
    qs_valid = qs_array[~np.isnan(qs_array)]  # Filter out NAN values
    W = qs_valid.mean()
    R = np.sqrt(OG_dataset.shape[0]) * (qs_valid - W)
    alpha = 0.05
    L = np.quantile(R, alpha/2) 
    U = np.quantile(R, 1 - alpha/2)

    BQS = L

    # Average ARI
    ARI_array = np.array(ARI_list)
    ARI_mean = np.nanmean(ARI_array)

    # Jaccard scores per cluster
    jdf = pd.DataFrame(jaccard_list)
    jaccard_mean = jdf.mean().to_dict()
    jaccard_std = jdf.std().to_dict()

    results_dict = {
        'K': OG_clusterID,
        'ARI': ARI_mean,
        'QS_mean': W,
        'L_ci': L,
        'U_ci': U,
        'all_QS': qs_array,
        '%-discarded': proportion_empty,
        'duration': f'{time.time()-start:.2f}s',
        'b': n_bootstrap,
        'Jaccard': jaccard_mean
    }

    print(pd.DataFrame([results_dict])[['K', 'ARI', 'QS_mean', 'L_ci', 'U_ci', '%-discarded', 'duration', 'b', 'Jaccard']])
    #print(f' Jaccard Mean(std): {jaccard_mean} ({jaccard_std})') 

    return OG_clusterID, round(BQS,5), results_dict

In [10]:
#!cat debug.log

In [55]:
BQS = []
bqs_list = []

parent_folder = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/Bootstrap Data"

for folder in os.listdir(parent_folder):
     #if folder.startswith("GSEM"):   # All folders but Skip hidden directories
     if folder.startswith("GSEM Gauss"):  # if you only want to run 1 algorithm subfolders
        print(folder)
        folder_path = os.path.join(parent_folder, folder)
        for subfolder in os.listdir(folder_path):
            if subfolder.startswith("."):   #Skip hidden directories
                continue
            subfolder_path = os.path.join(folder_path, subfolder)
            print(subfolder)
            if os.path.isdir(subfolder_path):
                total_files = len([f for f in os.listdir(subfolder_path) if os.path.isfile(os.path.join(subfolder_path, f)) and f.endswith(".dta")]) #List of file names in the folder
                failed_files = len([f for f in os.listdir(subfolder_path) if os.path.isfile(os.path.join(subfolder_path, f)) and f.endswith("_FAILED.dta")])
                p_failed = failed_files/total_files*100

                # If %discarded > 25% do not compute QS
                if p_failed > 25:
                    results_dict = {
                            'K': subfolder,
                            'ARI': "",
                            'QS_mean': "",
                            'L_ci': "",
                            'U_ci': "",
                            '%-discarded': p_failed,
                            'duration': "",
                            'b':"", 
                            'Jaccard':""
                    }
                    bqs_list.append(pd.DataFrame([results_dict])[['K', 'ARI', 'QS_mean', 'L_ci', 'U_ci', '%-discarded', 'duration', 'b', 'Jaccard']])
                    continue
                    
                else:
                    OG_clusterID, qs_bootstrapped, results_dict = compute_BQS_ARI_inParallel(subfolder_path, OG_dataset, cluster_columns)
                    BQS.append({'K': OG_clusterID, 'BQS':qs_bootstrapped})
                    bqs_list.append(pd.DataFrame([results_dict])[['K', 'ARI', 'QS_mean', 'L_ci', 'U_ci', '%-discarded', 'duration', 'b', 'Jaccard']])
                    #results[subfolder] = {"BQS": BQS, "results_dict": results_dict}
                   
os.system('say "BQS Computed!"')
print('Done computing BQS')

#Store Data
#Mark dataset with computation date.time
timestamp = datetime.now().strftime("%Y%m%d_%H:%M:%S")
bqs_results = pd.concat(bqs_list, ignore_index=True)
bqs_results['timestamp'] = timestamp

# Store
file_name = f'PsAID_BQS_MixtureModels_{timestamp}'

# Ensure the folder exists
os.makedirs(data_storage, exist_ok=True)

# Save
#bqs_results.to_csv(f'{data_storage}/{file_name}.csv', index=False)
print(f' Dataset saved to {data_storage}/{file_name}.csv')
bqs_results

GSEM Gauss
gsemGaussK6_bootstrap1000
n_bootstrap: 1000


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.0s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:    4.6s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:    6.8s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:    9.7s
[Parallel(n_jobs=-1)]: Done 632 tasks      | elapsed:   13.5s
[Parallel(n_jobs=-1)]: Done 866 tasks      | elapsed:   17.8s
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:   20.2s finished


                        K       ARI  QS_mean      L_ci       U_ci  \
0  clusterID_gsemGauss_k6  0.758752  -8.6936 -20.11117  19.170049   

   %-discarded duration     b  \
0          0.0   20.24s  1000   

                                             Jaccard  
0  {0: 0.7676728564369152, 1: 0.7962770203636919,...  
Done computing BQS
 Dataset saved to /Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/Quadratic Scores/PsAID_BQS_MixtureModels_20260731_18:39:10.csv


,K,ARI,QS_mean,L_ci,U_ci,%-discarded,duration,b,Jaccard,timestamp
0,clusterID_gsemGauss_k6,0.758752,-8.6936,-20.11117,19.170049,0.0,20.24s,1000,"{0: 0.7676728564369152, 1: 0.7962770203636919,...",20260731_18:39:10


In [66]:
for v in bqs_results['Jaccard']: print(v)

{0: 0.7676728564369152, 1: 0.7962770203636919, 2: 0.707858553578918, 3: 0.8757139251392078, 4: 0.5875429052212592, 5: 0.7008578773620109}
